In [1]:
import typing as T
import pickle
import json
import os
import pathlib
import pathlib as P
import sys
import pandas as pd
import itertools as it
import functools as ft
import operator as opr
import collections as clt
import math
from math import pi
import re

In [2]:
prj_root = P.Path("__file__").absolute().parent.parent.parent
if (p := str(prj_root)) not in sys.path:
  sys.path.append(p)

In [3]:
import util.metrics as um
import sklearn.metrics as metrics

In [4]:
import torch as th
import numpy as np
import scipy as sci
import scipy.stats as stats
import scipy.sparse as sp
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as colors
import scipy.stats as ss
import seaborn as sns
from tqdm import tqdm

In [5]:
ns = ['cc', 'mf', 'bp']
ontology_lst = ["cellular_component",
                "molecular_function",
                "biological_process"]

In [6]:
data_path = [f"/data0/shaojiangyi/pprogo-flg-1/data/data-netgo/{x}_data.pkl"
             for x in ["train", "valid", "test"]]
prot_data = pd.concat([pd.read_pickle(x) for x in data_path], ignore_index=True)

In [7]:
prot_labels = []
for n in ns:
    y_label = prj_root / "data" / n / "label.pkl"
    with open(y_label, "rb") as f:
        labels = pickle.load(f)
    tmp = clt.defaultdict(list)
    for k, v in zip(labels["protein"], labels["go"]):
        tmp[k].append(v)
    prot_labels.append(tmp)

In [8]:
len(prot_labels[2])

91443

In [9]:
label_dir = P.Path("/data0/zhaojianxiang/data/")
curr_labels = []
for n in ns:
    label_path = label_dir / n / "go_list.txt"
    with open(label_path, "r") as f:
        labels = f.read().splitlines()
    curr_labels.append(labels)
namespace_terms = dict(zip(ontology_lst, curr_labels))

In [10]:
# get protein name id
name_path = prj_root / "data" / "protein_name.txt"
with open(name_path, "r") as f:
    prot_lst = f.read().splitlines()# load prot names

In [11]:
# convert protein idx and label idx to protein name and go term respectively
for i, n in enumerate(ontology_lst):
    prot_labels[i] = {
        prot_lst[k]: [namespace_terms[n][x] for x in v]
        for k, v in prot_labels[i].items()
    }

In [12]:
# load predictions
pred_path = [prj_root / "data" / "prediction" / f"hgat_predictions_{x}.pt"
             for x in ["train", "valid", "test"]]
protpred = [torch.load(p) for p in pred_path]

In [13]:
prot_matrix = [[th.stack(list(p.values())).transpose(0,1).detach().cpu().numpy()
                for p in pred] for pred in protpred]

In [14]:
# get fmax of all the proteins
split_type = ["train", "valid", "test"]
perfinfo_lst = clt.defaultdict(list)
for (i, split), (j, n) in tqdm(it.product(enumerate(split_type),
                                     enumerate(ns)),total=len(split_type)*len(ns),ncols=120):
    tp = prot_matrix[i][j]
    (fmax_value, ths), auprc = um.f1_max(*tp, 
                                        need_threshold=True,
                                        no_empty_labels=True,
                                        no_zero_classes=True,
                                        auprc=True)
    perfinfo_lst["split"].append(split)
    perfinfo_lst["ontology"].append(n)
    perfinfo_lst["fmax"].append(fmax_value)
    perfinfo_lst["threshold"].append(ths)
    perfinfo_lst["auprc"].append(auprc)

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [28:35<00:00, 190.59s/it]


In [15]:
perfinfo_df = pd.DataFrame(perfinfo_lst)

In [16]:
perfinfo_df

,split,ontology,fmax,threshold,auprc
0,train,cc,0.708816,0.24,0.771653
1,train,mf,0.686706,0.15,0.721170
2,train,bp,0.459297,0.20,0.449769
3,valid,cc,0.708239,0.24,0.771157
4,valid,mf,0.686478,0.15,0.721573
5,valid,bp,0.460883,0.20,0.451502
6,test,cc,0.698021,0.33,0.736112
7,test,mf,0.708337,0.15,0.732521
8,test,bp,0.406361,0.25,0.379626


In [17]:
threshold_nested = perfinfo_df.groupby('split')['threshold'].apply(list).tolist()

In [18]:
prot_matbi = [[np.stack([p[0], np.where(p[1] > threshold_nested[i][j], 1., 0.)])
               for j, p in enumerate(pred)] for i, pred in enumerate(prot_matrix)]

In [19]:
prot_matbi = [[th.tensor(p) for p in pred] for pred in prot_matbi]

In [20]:
protpred_bi = [[{n: x for n, x in zip(protpred[i][j].keys(),
                                  p.transpose(0,1))} for j, p in enumerate(pred)] for i, pred in enumerate(prot_matbi)]

In [21]:
# save binary prediction
bi_path = [prj_root / "data" / "prediction" / f"hgat_predictions_{x}_bi.pt"
             for x in ["train", "valid", "test"]]
for i, save_path in enumerate(bi_path):
    th.save(protpred_bi[i], save_path)

In [24]:
def f1_max_bi(targs, preds_bi):
    tp = (preds_bi * targs).sum(1)
    tp_fp = preds_bi.sum(1)
    tp_fn = targs.sum(1)

    idx= np.where(tp_fp != 0)
    tp_fp = tp_fp[idx]
    precision = (tp[idx] / tp_fp
    if tp_fp.shape[0] != 0
    else np.asarray([0.])).mean()

    # when no_empty_label is False
    idx = np.where(tp_fn != 0)
    tp_fn = tp_fn[idx]
    recall = (tp[idx] / tp_fn
    if tp_fn.shape[0] != 0
    else np.asarray([0.])).mean()

    if ((denom := precision + recall) == 0.0):
        denom = 1.0
    fmax_value = 2 * precision * recall / denom
    
    return fmax_value

In [48]:
def get_perf_report(targs, preds, preds_bi):
    targs = targs[None, :]
    preds = preds[None, :]
    
    preds_bi = preds_bi[None, :]
    
    targs = targs.cpu().detach().numpy()
    preds = preds.cpu().detach().numpy()
    preds_bi = preds_bi.cpu().detach().numpy()
    _, auprc_value = um.f1_max(targs, preds, auprc=True)
    fmax_value = f1_max_bi(targs, preds_bi)
    # roc_value = um.roc_auc_score(targs, preds)
    return {"fmax": fmax_value, "auprc": auprc_value,
            # "roc": roc_value
            }

In [49]:
split_type = ["train", "valid", "test"]
perf_data = clt.defaultdict(list)
for i, s in enumerate(split_type):
    for j, n in enumerate(ns):
        for name, pred in tqdm(protpred[i][j].items(),
                               total=len(protpred[i][j]), ncols=120):
            perf_data["split"].append(s)
            perf_data["ontology"].append(n)
            perf_data["proteins"].append(name)
            report = get_perf_report(pred[0], pred[1], 
                                     protpred_bi[i][j][name][1])
            for k, v in report.items():
                perf_data[k].append(v)

100%|████████████████████████████████████████████████████████████████████████████████| 491/491 [00:04<00:00, 102.90it/s]


In [50]:
perf_df = pd.DataFrame(perf_data)

In [51]:
perf_df

,split,ontology,proteins,fmax,auprc
0,train,cc,Q6R327,0.476190,0.610485
1,train,cc,A4VCH8,0.000000,0.500000
2,train,cc,F4HYR6,0.000000,0.500000
3,train,cc,P34462,0.000000,0.500000
4,train,cc,Q9HGZ6,0.000000,0.500000
...,...,...,...,...,...
372011,test,bp,Q9VT33,0.160000,0.120250
372012,test,bp,Q9VYA0,0.120000,0.110857
372013,test,bp,Q9W5X1,0.298507,0.327778
372014,test,bp,Q9Y320,0.153846,0.142901


In [52]:
ont_map = {"cc":0, "mf": 1, "bp": 2}
perf_df["annotations"] = [prot_labels[ont_map[i]].get(x) for i, x in zip(perf_df["ontology"], perf_df["proteins"])]

In [53]:
# save result
save_dir = prj_root / "data" / "prediction"
save_path = save_dir / "hgat_performance_2.pkl"
perf_df.to_pickle(save_path)